# 📊 Plantilla Maestra de Análisis Exploratorio de Datos (EDA)

Esta plantilla proporciona un framework exhaustivo de **18 pasos** para abordar el análisis de cualquier conjunto de datos tabular. Todo el código está contenido aquí mismo (Self-Contained) para que puedas ejecutarlo celda por celda.

## 1. Configuración e Importación de Librerías
Configuramos el entorno y cargamos las librerías necesarias.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="mako")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11

print("✅ Librerías importadas correctamente.")

## 2. Carga del Dataset
Cargaremos el dataset de ejemplo "Titanic". Para usar tu propio archivo, descomenta la línea de `pd.read_csv`.

In [ ]:
data_path = "../../data/customer_support_data.csv"
df = pd.read_csv(data_path)
print("✅ Dataset cargado correctamente.")


## 3. Snapshot de los Datos
Revisamos rápidamente las dimensiones y las primeras filas.

In [ ]:
print(f"Dimensión del Dataset: {df.shape[0]:,} Filas, {df.shape[1]} Columnas")
display(df.head())
display(df.sample(3)) # Muestra aleatoria para evitar sesgos iniciales

## 4. Estructura y Metadata
Validamos los tipos de datos asignados automáticamente por Pandas.

In [ ]:
df.info()

## 5. Diccionario Automático de Variables
Separamos automáticamente las variables numéricas de las categóricas.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"🔢 Variables Numéricas ({len(num_cols)}): {num_cols}")
print(f"🔠 Variables Categóricas ({len(cat_cols)}): {cat_cols}")

## 6. Evaluación de Calidad: Valores Nulos
Identificamos columnas con datos faltantes.

In [ ]:
nulos = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos["Cantidad"] > 0].sort_values("Porcentaje (%)", ascending=False)

if nulos.empty:
    print("✅ No hay valores nulos.")
else:
    display(nulos)

## 7. Tratamiento de Valores Nulos
Ejemplo de imputación: usamos la mediana para números y la moda para categorías.

In [ ]:
for col in nulos.index:
    if col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])
print("✅ Nulos imputados.")

## 8. Identificación y Limpieza de Duplicados

In [ ]:
duplicados = df.duplicated().sum()
print(f"Registros duplicados encontrados: {duplicados}")
if duplicados > 0:
    df = df.drop_duplicates()
    print("✅ Duplicados eliminados.")

## 9. Estadísticas Descriptivas (Numéricas)
Medidas de tendencia central y dispersión.

In [ ]:
display(df[num_cols].describe().T)

## 10. Estadísticas Descriptivas (Categóricas)
Frecuencias y cardinalidad de clases.

In [ ]:
if cat_cols:
    display(df[cat_cols].describe(include="all").T)

## 11. Análisis Univariado Numérico: Distribuciones e Histogramas

In [ ]:
import math

n_cols = 3
n_rows = math.ceil(len(num_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="teal", bins=25)
    axes[i].set_title(f"Distribución de {col}")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 12. Análisis Univariado Categórico: Barplots

In [ ]:
n_rows_cat = math.ceil(len(cat_cols) / n_cols)
if cat_cols:
    fig, axes = plt.subplots(n_rows_cat, n_cols, figsize=(18, n_rows_cat * 4))
    axes = axes.flatten()
    
    for i, col in enumerate(cat_cols):
        sns.countplot(y=df[col], ax=axes[i], palette="viridis", order=df[col].value_counts().index[:10])
        axes[i].set_title(f"Frecuencia de {col}")
    
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.show()

## 13. Detección Matemática de Valores Atípicos (Outliers via IQR)

In [ ]:
outliers_dict = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))].shape[0]
    if outliers > 0:
        outliers_dict[col] = outliers

outliers_df = pd.DataFrame(list(outliers_dict.items()), columns=["Columna", "Cant. Outliers"])
outliers_df["% Outliers"] = (outliers_df["Cant. Outliers"] / len(df) * 100).round(2)
display(outliers_df.sort_values("% Outliers", ascending=False))

## 14. Visualización de Outliers (Boxplots)

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x=df[col], ax=axes[i], color="lightseagreen")
    axes[i].set_title(f"Boxplot de {col}")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 15. Análisis Bivariado: Correlación Numérica (Heatmap)

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df[num_cols].corr(method="pearson")
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Matriz de Correlación de Pearson")
plt.show()

## 16. Análisis Bivariado: Numérica vs Categórica
Visualizamos cómo varían las numéricas según una categoría (Ej: sobrevivientes). Define la variable TARGET.

In [ ]:
target = "survived" # Cambia esto por tu variable objetivo

if target in cat_cols and len(num_cols) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    sns.violinplot(data=df, x=target, y=num_cols[0], ax=axes[0], palette="muted")
    axes[0].set_title(f"Distribución de {num_cols[0]} por {target}")
    
    sns.boxplot(data=df, x=target, y=num_cols[1], ax=axes[1], palette="muted")
    axes[1].set_title(f"Distribución de {num_cols[1]} por {target}")
    
    plt.tight_layout()
    plt.show()

## 17. Feature Engineering Básica
Ejemplo: Creamos una métrica combinada si es aplicable. (En Titanic: Tamaño de familia).

In [ ]:
if "sibsp" in df.columns and "parch" in df.columns:
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    print("Variable family_size creada con éxito.")
    
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x="family_size", hue="survived", palette="crest")
    plt.title("Supervivencia vs Tamaño de Familia")
    plt.show()

## 18. Conclusiones y Preparación para Machine Learning
Resumen general para continuar con la etapa de modelado.

In [ ]:
print("🎯 EDA COMPLETADO.")
print("Siguientes pasos recomendados:")
print("1. Escalar variables numéricas usando StandardScaler.")
print("2. Aplicar One-Hot Encoding a variables categóricas nominales.")
print("3. Balancear las clases de la variable objetivo si aplica.")
print("4. Seleccionar un modelo base de Machine Learning (Baseline).")